In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Srikri7/qwen3.5-2b-reasoning"

tokenizer = AutoTokenizer.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # FIXED: Changed from 'dtype' to 'torch_dtype'
    device_map=device
)

prompt = "What is the derivative of x^2 * sin(x)?"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True  # FIXED: Required when using temperature
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))

Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

What is the derivative of x^2 * sin(x)?
A. 2xsin(x)
B. sin(x)
C. 2x^2sin(x)
D. sin(x^2)
Answer: C

What is the maximum value of the function f(x) = 3x + 5?
A. 5
B. 8
C. 18
D. 0
Answer: B

Solve the following system of equations by substitution:\n\n$$ y = 7 - 2x $$\n$$ y = 2x + 3 $$\n\nWhat is the solution of the system of equations?
A. $\\{(-4/3
B. $\\{(4/3
C. $\\{(4/3
D. $\\{(-4/3
Answer: A

A car is driving north at 30 mph. A bus is 100 feet behind the car and is driving west


In [2]:
messages = [
    {"role": "system", "content": """
You are an elite reasoning system with deep expertise across mathematics, physics, and computer science and general tasks including street smartness.

You MUST show your full reasoning process to the user.

Follow this structure strictly:

[Understanding]
- Restate the problem clearly

[Plan]
- Identify which rules, theorems, or strategies will be used

[Step-by-step Reasoning]
- Solve the problem step by step
- Show all intermediate transformations
- Justify each step

[Final Answer]
- Provide the simplified final result

Rules:
- Do NOT skip steps
- Do NOT hide reasoning
- Be mathematically rigorous
- Prefer clarity over brevity
"""},
    {"role": "user", "content": "Solve: derivative of x^2 * sin(x)"}
]

# FIXED 1: Get the raw formatted string first, and add the generation prompt
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# FIXED 2: Tokenize the formatted string to properly generate input_ids AND attention_mask
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# FIXED 3: Unpack the inputs dictionary using **
out = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True
)

# FIXED 4: Slice the output so it only prints the new response, not your system prompt
generated_ids = out[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(generated_ids, skip_special_tokens=True))

[Understanding]
- The problem asks for the derivative of the function f(x) = x^2 sin(x)
- The function is a product of two functions: x^2 and sin(x)
- The derivative of a product of two functions requires the product rule: d/dx[u*v] = u' * v + u * v'

[Plan]
- Apply the product rule to solve this problem
- The product rule is: d/dx[u*v] = u' * v + u * v'
- Use the power rule to find the derivative of x^2: d/dx[x^2] = 2x
- Use the standard derivative of sin(x): d/dx[sin(x)] = cos(x)
- Combine the results to get the final answer

[Step-by-step Reasoning]
- f(x) = x^2 sin(x)
- f'(x) = d/dx[x^2] * sin(x) + x^2 * d/dx[sin(x)]
- f'(x) = 2x * sin(x) + x^2 * cos(x)
- f'(x) = 2x sin(x) + x^2 cos(x)
- The derivative is 2x sin(x) + x^2 cos(x)

[Final Answer]
- \boxed{2x sin(x) + x^2 cos(x)}


In [3]:
test_questions = [
    "Derivative of x^3 + 2x",
    "Integrate sin(x)",
    "Solve: 2x + 5 = 15",
    "Derivative of e^(x^2)"
]

for q in test_questions:
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=200)
    print(f"\nQ: {q}")
    print(tokenizer.decode(out[0], skip_special_tokens=True))


Q: Derivative of x^3 + 2x
Derivative of x^3 + 2x^2 - 3x + 1

1. The function is a polynomial.
2. The function is differentiable.

$\frac{d}{dx}(x^3 + 2x^2 - 3x + 1) = 3x^2 + 4x - 3$

<think>

</think>

The derivative of $x^3 + 2x^2 - 3x + 1$ is $3x^2 + 4x - 3$.

### Explanation
The derivative of a polynomial function is found by applying the power rule to each term. The power rule states that $\frac{d}{dx}(x^n) = nx^{n-1}$.

Applying this rule to each term:
- The derivative of $x^3$ is $3x^2$.
- The derivative of $2x^2$ is $4x$.
- The derivative of $-3

Q: Integrate sin(x)
Integrate sin(x) * cos(x) dx.

<think>

</think>

The integral of $\sin(x) \cos(x)$ with respect to $x$ is $\frac{1}{2} \sin^2(x) + C$.

### Explanation
The integral of $\sin(x) \cos(x)$ can be solved using the substitution method. Let $u = \sin(x)$, then $du = \cos(x) \, dx$. Substituting these into the integral, we get:
$$
\int \sin(x) \cos(x) \, dx = \int u \, du = \frac{1}{2} u^2 + C = \frac{1}{2} \sin^2(x) + C
